In [ ]:
import pandas as pd
import os
import re
from pathlib import Path



import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer




nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("stopwords")


In [ ]:
def find_project_root(start_path, marker="data"):
    path = Path(start_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

In [ ]:
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

In [ ]:
fli_words_base = {
    "will", "expect", "expects", "expected",
    "anticipate", "anticipates", "anticipated",
    "plan", "plans", "planned",
    "forecast", "forecasts",
    "estimate", "estimates",
    "project", "projects", "projected",
    "intend", "intends", "intended",
    "aim", "aims",
    "target", "targets",
    "outlook"
}

# Forward-looking dictionary from Li (2010)
fli_words = {
    "will","should","can","could","may","might",
    "expect","anticipate","believe","plan",
    "intend","seek","project","forecast",
    "objective","goal"
}

# legal boilerplate words
legal_words = {
    "herein","hereinafter","hereof",
    "hereon","hereto","theretofore","therein",
    "thereof","thereon"
}

disclaimer_words = {
    "litigation",
    "reform",
    "safe harbor",
    "no assurance",
    "actual results",
    "undertake no obligation"
}

# past expectation patterns
past_patterns = re.compile(
    r"\b(was|were|had|had been|has|has been)\s+(expect|anticipat|forecast|project|believe)\b" #lemmatisation applied
)



In [ ]:
def split_sentences(text):
    return re.split(r'(?<=[.!?])\s+', text)

def is_all_caps(sentence):
    letters = re.sub(r'[^A-Za-z]', '', sentence)
    return letters.isupper() and len(letters) > 5


In [ ]:
stop_words = set(stopwords.words("english"))

# keep modal verbs used in forward-looking statements
stop_words = stop_words - {"will", "would", "should", "may", "might"}

lemmatizer = WordNetLemmatizer()

lemma_dict = set(lemmatizer.lemmatize(w, pos="v") for w in fli_words)

def tokenize(text):

    text = text.lower()

    # remove speaker tags
    text = re.sub(r"\b(operator|executive|analyst)\s*[:\-]", "", text)

    # remove punctuation / numbers
    text = re.sub(r"[^a-z\s]", " ", text)

    tokens = text.split()

    # remove stopwords
    tokens = [t for t in tokens if t not in stop_words]

    # lemmatize verbs
    tokens = [lemmatizer.lemmatize(t, pos="v") for t in tokens]

    return tokens

In [ ]:
def compute_fli(df, lemma_dict):

    results = []

    for row in df.itertuples(index=False):
        text = row.transcript_text
        transcriptid = row.transcriptid
        companyid = row.companyid
        

        sentences = split_sentences(text)

        total_sentences = 0
        fli_sentences = []

        for s in sentences:

            # remove ALL CAPS sentences
            if is_all_caps(s):
                continue

            s_lower = s.lower()

            # remove disclaimers / legal
            if any(word in s_lower for word in disclaimer_words):
                continue

            if any(word in s_lower for word in legal_words):
                continue

            # remove past expectation phrases
            if past_patterns.search(s_lower):
                continue

            # remove "was"
            if " was " in f" {s_lower} ":
                continue

            total_sentences += 1

            tokens = tokenize(s)

            if any(t in lemma_dict for t in tokens):
                fli_sentences.append(s)

        # compute score
        if total_sentences == 0:
            fli_score = 0
        else:
            fli_score = len(fli_sentences) / total_sentences

        results.append({
            "transcriptid": transcriptid,
            "companyid": companyid,
            "fli_score": fli_score,
            "n_sentences": total_sentences,
            "n_fli_sentences": len(fli_sentences)
        })

    return pd.DataFrame(results)

In [ ]:
sample = pd.read_csv(DATA_RAW/ 'transcripts_final.csv')

In [ ]:
result = compute_fli(sample, fli_words)

In [ ]:
result.columns

In [ ]:
result['fli_score'].describe()

In [ ]:
result["fli_score"].hist(bins=30)

In [ ]:
result[["fli_score","n_sentences"]].corr()

In [ ]:
result.to_csv(DATA_PROCESSED/'dictionary_output.csv')